# 07 多个参数的整体梯度长度

## 知识点标注
- 行业状态：常见工程实践。
- 现实用途：模型有大量参数时，`clip_grad_norm_` 会把所有参数的梯度视为一支高维箭头。
- 本课类型：本地概念演示，不使用 GPU，不训练 MokioMind。

本课只看两个参数，先把高维规则缩小成能画出来的二维例子。

## 两个参数就是一支二维箭头

如果两个参数的梯度是 `[3, 4]`：

```text
       ↑ w2 梯度 = 4
       │      ● (3,4)
       │     /
       │    /  L2 长度 = 5
───────┼────────→ w1 梯度 = 3
```

计算：`sqrt(3² + 4²) = 5`。如果最大允许长度是 `2.5`，缩放比例就是 `2.5 / 5 = 0.5`，所以新梯度是 `[1.5, 2.0]`。方向不变，箭头整体缩短一半。

In [1]:
import math
import torch

grad = torch.tensor([3.0, 4.0])
old_norm = torch.linalg.vector_norm(grad)
max_norm = 2.5
scale = max_norm / old_norm
new_grad = grad * scale

print('原梯度：', grad.tolist())
print('原范数：', old_norm.item())
print('缩放比例：', scale.item())
print('裁剪后梯度：', new_grad.tolist())
print('裁剪后范数：', torch.linalg.vector_norm(new_grad).item())

原梯度： [3.0, 4.0]
原范数： 5.0
缩放比例： 0.5
裁剪后梯度： [1.5, 2.0]
裁剪后范数： 2.5


### 先预测

1. `[3,4]` 的整体范数是多少？
2. 如果最大范数是 `2.5`，两个分量会不会分别被裁成 `2.5`？
3. 裁剪后方向会改变吗？

In [2]:
w1 = torch.tensor([1.0], requires_grad=True)
w2 = torch.tensor([1.0], requires_grad=True)

# 这个 loss 在 w1=w2=1 时产生梯度 [3,4]
loss = 1.5 * w1.square() + 2.0 * w2.square()
loss.backward()
print('backward 后参数：', w1.item(), w2.item())
print('backward 后梯度：', w1.grad.item(), w2.grad.item())

old_norm = torch.nn.utils.clip_grad_norm_([w1, w2], max_norm=2.5)
print('裁剪前整体范数：', old_norm.item())
print('裁剪后梯度：', w1.grad.item(), w2.grad.item())
print('裁剪后整体范数：', math.sqrt(w1.grad.item() ** 2 + w2.grad.item() ** 2))

backward 后参数： 1.0 1.0
backward 后梯度： 3.0 4.0
裁剪前整体范数： 5.0
裁剪后梯度： 1.4999996423721313 1.9999995231628418
裁剪后整体范数： 2.4999994039535522


## 这和模型的海量参数有什么关系？

真实模型有很多参数张量，不会真的把它们拼成一个大数组再保存；PyTorch 会逻辑上计算所有参数梯度的整体范数。可以把它想成：

```text
所有参数的梯度分量
        ↓ 平方、求和、开平方
一根高维梯度箭头的长度
        ↓ 超过阈值就统一缩放
交给 optimizer.step() 更新参数
```

因此它不是把每个参数单独限制在 `[-max_norm, max_norm]`，而是限制所有参数合起来的总长度。

## 对应 MokioMind 真实代码

项目中对应的是：

```python
torch.nn.utils.clip_grad_norm_(
    model.parameters(),  # 模型的全部参数
    args.grad_clip,      # 最大整体范数，默认 1.0
)
scaler.step(optimizer)   # 使用裁剪后的梯度更新参数
```

`model.parameters()` 会提供模型所有参数；API 内部按整体范数处理这些参数的梯度。真实位置：[trainer/train_pretrain.py:71](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:71)。阈值默认值在 [trainer/train_pretrain.py:165](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:165)。

## 验收题

1. 两个参数梯度 `[3,4]` 的 L2 范数怎么算？
2. 最大范数是 `2.5` 时，为什么裁剪后是 `[1.5,2.0]`？
3. `clip_grad_norm_(model.parameters(), ...)` 限制的是每个参数各自的长度，还是所有参数梯度的整体长度？
4. 裁剪后真正修改模型参数的是哪一行？